# Stage 3 - RL fine-tune ViT5 checkpoint trên domain Y tế

Notebook này tiếp tục từ output Stage 2:

- SFT model: `/content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/model_for_rl`
- Stage 2 cuối: ROUGE-L 79.75, BERTScore 95.19, number recall 87.38, number hallucination 3.67, length compliance 43.81, length ratio mean 0.2219.

Mục tiêu RL: tăng factuality với số liệu và length compliance, đồng thời giữ chất lượng ROUGE. Notebook dùng reward-shaped policy gradient nhẹ thay vì TRL/PPO để tránh xung đột version Colab.


## 1. Setup

In [1]:
import sys, subprocess, importlib.metadata as md

PACKAGES = [
    'transformers==4.46.3',
    'datasets==3.1.0',
    'evaluate==0.4.3',
    'accelerate==1.1.1',
    'rouge-score==0.1.2',
    'bert-score==0.3.13',
    'sentencepiece==0.2.0',
    'tqdm',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', '--no-cache-dir', *PACKAGES])

for pkg in ['transformers', 'datasets', 'evaluate', 'accelerate', 'rouge-score', 'bert-score', 'sentencepiece']:
    print(f'{pkg}: {md.version(pkg)}')


transformers: 4.46.3
datasets: 3.1.0
evaluate: 0.4.3
accelerate: 1.1.1
rouge-score: 0.1.2
bert-score: 0.3.13
sentencepiece: 0.2.0


In [2]:
import os, re, json, math, gc, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    get_linear_schedule_with_warmup,
    set_seed,
)
from rouge_score import rouge_scorer

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')


CUDA: True | NVIDIA A100-SXM4-40GB


## 2. Paths & config

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/data')
STAGE2_DIR = Path('/content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint')
SFT_MODEL_DIR = STAGE2_DIR / 'model_for_rl'
STAGE2_RESULTS = STAGE2_DIR / 'results' / 'final_results_stage2.json'

RL_DIR = Path('/content/drive/MyDrive/NLP_Final/Output stage 3 vit5 rl')
RL_CHECKPOINT_DIR = RL_DIR / 'checkpoints'
RL_BEST_DIR = RL_DIR / 'best_model'
RL_MODEL_FOR_NEXT_DIR = RL_DIR / 'model_for_next_stage'
RL_RESULTS_DIR = RL_DIR / 'results'
for d in (RL_CHECKPOINT_DIR, RL_BEST_DIR, RL_MODEL_FOR_NEXT_DIR, RL_RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

required = [
    DATA_DIR / 'train' / 'dataset.jsonl',
    DATA_DIR / 'train' / 'train_augmented.jsonl',
    DATA_DIR / 'val' / 'dataset.jsonl',
    DATA_DIR / 'test' / 'dataset.jsonl',
    SFT_MODEL_DIR / 'config.json',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Thiếu file:\n' + '\n'.join(missing))

print('DATA_DIR       :', DATA_DIR)
print('SFT_MODEL_DIR  :', SFT_MODEL_DIR)
print('RL_DIR         :', RL_DIR)
print('RL_BEST_DIR    :', RL_BEST_DIR)


Mounted at /content/drive
DATA_DIR       : /content/drive/MyDrive/data
SFT_MODEL_DIR  : /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/model_for_rl
RL_DIR         : /content/drive/MyDrive/NLP_Final/Output stage 3 vit5 rl
RL_BEST_DIR    : /content/drive/MyDrive/NLP_Final/Output stage 3 vit5 rl/best_model


In [4]:
CONFIG = {
    'USE_AUGMENTED_TRAIN': False,
    'MAX_INPUT_LEN': 768,
    'MAX_TARGET_LEN': 320,
    'TRAIN_BATCH': 1,
    'GRAD_ACCUM': 8,
    'EVAL_BATCH': 1,
    'EPOCHS': 2,
    'LR': 8e-6,
    'WARMUP_RATIO': 0.05,
    'MAX_GRAD_NORM': 1.0,
    'KL_COEF': 0.02,
    'BASELINE_MOMENTUM': 0.9,
    'DO_SAMPLE': True,
    'TOP_P': 0.92,
    'TEMPERATURE': 0.85,
    'NUM_BEAMS_TRAIN': 1,
    'NUM_BEAMS_EVAL': 2,
    'NO_REPEAT_NGRAM': 3,
    'LENGTH_PENALTY': 1.0,
    'W_ROUGE_L': 0.35,
    'W_NUMBER_RECALL': 0.25,
    'W_LENGTH_SCORE': 0.25,
    'W_HALLUCINATION': 0.45,
    'W_EMPTY': 0.30,
    'TARGET_RATIO': 0.225,
    'LEN_LO': 0.20,
    'LEN_HI': 0.25,
    'VAL_REWARD_SAMPLES': 32,
}
CONFIG


{'USE_AUGMENTED_TRAIN': False,
 'MAX_INPUT_LEN': 768,
 'MAX_TARGET_LEN': 320,
 'TRAIN_BATCH': 1,
 'GRAD_ACCUM': 8,
 'EVAL_BATCH': 1,
 'EPOCHS': 2,
 'LR': 8e-06,
 'WARMUP_RATIO': 0.05,
 'MAX_GRAD_NORM': 1.0,
 'KL_COEF': 0.02,
 'BASELINE_MOMENTUM': 0.9,
 'DO_SAMPLE': True,
 'TOP_P': 0.92,
 'TEMPERATURE': 0.85,
 'NUM_BEAMS_TRAIN': 1,
 'NUM_BEAMS_EVAL': 2,
 'NO_REPEAT_NGRAM': 3,
 'LENGTH_PENALTY': 1.0,
 'W_ROUGE_L': 0.35,
 'W_NUMBER_RECALL': 0.25,
 'W_LENGTH_SCORE': 0.25,
 'W_HALLUCINATION': 0.45,
 'W_EMPTY': 0.3,
 'TARGET_RATIO': 0.225,
 'LEN_LO': 0.2,
 'LEN_HI': 0.25,
 'VAL_REWARD_SAMPLES': 32}

## 3. Load data

In [5]:
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def to_pair(r):
    return {'id': r['id'], 'source': r['source'], 'summary': r['summary']}

train_file = DATA_DIR / 'train' / ('train_augmented.jsonl' if CONFIG['USE_AUGMENTED_TRAIN'] else 'dataset.jsonl')
train_raw = load_jsonl(train_file)
val_raw = load_jsonl(DATA_DIR / 'val' / 'dataset.jsonl')
test_raw = load_jsonl(DATA_DIR / 'test' / 'dataset.jsonl')

ds = DatasetDict({
    'train': Dataset.from_list([to_pair(r) for r in train_raw]),
    'val': Dataset.from_list([to_pair(r) for r in val_raw]),
    'test': Dataset.from_list([to_pair(r) for r in test_raw]),
})
print(ds)
print('train_file:', train_file)


DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'summary'],
        num_rows: 780
    })
    val: Dataset({
        features: ['id', 'source', 'summary'],
        num_rows: 94
    })
    test: Dataset({
        features: ['id', 'source', 'summary'],
        num_rows: 105
    })
})
train_file: /content/drive/MyDrive/data/train/dataset.jsonl


## 4. Load SFT policy and frozen reference

In [6]:
tokenizer = AutoTokenizer.from_pretrained(str(SFT_MODEL_DIR))
policy = AutoModelForSeq2SeqLM.from_pretrained(str(SFT_MODEL_DIR)).to(DEVICE)
ref_model = AutoModelForSeq2SeqLM.from_pretrained(str(SFT_MODEL_DIR)).to(DEVICE)

policy.gradient_checkpointing_enable()
policy.config.use_cache = False
ref_model.eval()
ref_model.requires_grad_(False)

print(f'Policy params: {sum(p.numel() for p in policy.parameters())/1e6:.1f}M')
print('gradient_checkpointing:', policy.is_gradient_checkpointing)
print('policy use_cache:', policy.config.use_cache)


Policy params: 226.0M
gradient_checkpointing: True
policy use_cache: False


In [7]:
def preprocess(batch):
    enc = tokenizer(
        batch['source'],
        max_length=CONFIG['MAX_INPUT_LEN'],
        truncation=True,
        padding=False,
    )
    enc['gold'] = batch['summary']
    enc['source_text'] = batch['source']
    enc['id'] = batch['id']
    return enc

tok_train = ds['train'].map(preprocess, batched=True, remove_columns=['source', 'summary'])

def rl_collate(features):
    model_inputs = [{'input_ids': f['input_ids'], 'attention_mask': f['attention_mask']} for f in features]
    batch = tokenizer.pad(model_inputs, padding=True, return_tensors='pt')
    batch['source_text'] = [f['source_text'] for f in features]
    batch['gold'] = [f['gold'] for f in features]
    batch['id'] = [f['id'] for f in features]
    return batch

train_loader = DataLoader(tok_train, batch_size=CONFIG['TRAIN_BATCH'], shuffle=True, collate_fn=rl_collate)
print('RL train batches:', len(train_loader))


Map:   0%|          | 0/780 [00:00<?, ? examples/s]

RL train batches: 780


## 5. Reward functions

Reward được thiết kế quanh điểm yếu còn lại sau Stage 2:

- `rougeL` giữ nội dung chính so với gold summary.
- `number_recall` thưởng giữ số liệu từ source.
- `number_hallucination` phạt sinh số không có trong source.
- `length_score` kéo tỷ lệ tóm tắt về 20-25%.


In [8]:
NUM_RE = re.compile(r'\d+(?:[.,]\d+)*')
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def extract_numbers(text):
    return set(NUM_RE.findall(text or ''))

def number_recall(src, pred):
    s = extract_numbers(src)
    if not s:
        return 1.0
    return len(s & extract_numbers(pred)) / len(s)

def number_hallucination(src, pred):
    p = extract_numbers(pred)
    if not p:
        return 0.0
    return len(p - extract_numbers(src)) / len(p)

def length_ratio(src, pred):
    return len((pred or '').split()) / max(len((src or '').split()), 1)

def length_score(src, pred, target=CONFIG['TARGET_RATIO']):
    ratio = length_ratio(src, pred)
    if ratio <= target:
        return max(0.0, 1.0 - (target - ratio) / target)
    else:
        return max(-0.5, 1.0 - 2.0 * (ratio - target) / target)

def length_compliance(src, pred, lo=CONFIG['LEN_LO'], hi=CONFIG['LEN_HI']):
    ratio = length_ratio(src, pred)
    return float(lo <= ratio <= hi)

def reward_one(src, gold, pred):
    pred = pred or ''
    r_l = rouge.score(gold, pred)['rougeL'].fmeasure
    n_recall = number_recall(src, pred)
    n_hallu = number_hallucination(src, pred)
    l_score = length_score(src, pred)
    empty_penalty = float(len(pred.strip()) == 0)

    src_has_numbers = len(extract_numbers(src)) > 0
    pred_has_numbers = len(extract_numbers(pred)) > 0
    number_avoidance_penalty = float(src_has_numbers and not pred_has_numbers) * 0.3  # thêm

    reward = (
        CONFIG['W_ROUGE_L'] * r_l
        + CONFIG['W_NUMBER_RECALL'] * n_recall
        + CONFIG['W_LENGTH_SCORE'] * l_score
        - CONFIG['W_HALLUCINATION'] * n_hallu
        - CONFIG['W_EMPTY'] * empty_penalty
        - number_avoidance_penalty  # thêm
    )
    return float(reward), {
        'rougeL': r_l,
        'number_recall': n_recall,
        'number_hallucination': n_hallu,
        'length_score': l_score,
        'length_compliance': length_compliance(src, pred),
        'length_ratio': length_ratio(src, pred),
        'number_avoidance': float(src_has_numbers and not pred_has_numbers),  # thêm để monitor
    }

def reward_batch(srcs, golds, preds):
    rewards, details = [], []
    for s, g, p in zip(srcs, golds, preds):
        r, d = reward_one(s, g, p)
        rewards.append(r)
        details.append(d)
    return rewards, details


## 6. RL utilities

In [9]:
def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def generate_for_rl(model, batch, do_sample=True, num_beams=1):
    gen_kwargs = dict(
        max_length=CONFIG['MAX_TARGET_LEN'],
        no_repeat_ngram_size=CONFIG['NO_REPEAT_NGRAM'],
        length_penalty=CONFIG['LENGTH_PENALTY'],
        early_stopping=True,
    )
    if do_sample:
        gen_kwargs.update(dict(
            do_sample=True,
            top_p=CONFIG['TOP_P'],
            temperature=CONFIG['TEMPERATURE'],
            num_beams=1,
        ))
    else:
        gen_kwargs.update(dict(
            do_sample=False,
            num_beams=num_beams,
        ))

    return model.generate(
        input_ids=batch['input_ids'].to(DEVICE),
        attention_mask=batch['attention_mask'].to(DEVICE),
        **gen_kwargs,
    )

def sequence_logprobs(model, batch, sequences):
    labels = sequences.clone().to(DEVICE)
    labels[labels == tokenizer.pad_token_id] = -100

    decoder_input_ids = model.prepare_decoder_input_ids_from_labels(labels)
    outputs = model(
        input_ids=batch['input_ids'].to(DEVICE),
        attention_mask=batch['attention_mask'].to(DEVICE),
        decoder_input_ids=decoder_input_ids,
        use_cache=False,
    )
    log_probs = F.log_softmax(outputs.logits, dim=-1)
    target = labels.masked_fill(labels == -100, 0)
    token_log_probs = log_probs.gather(-1, target.unsqueeze(-1)).squeeze(-1)
    mask = (labels != -100).float()
    seq_log_probs = (token_log_probs * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
    return seq_log_probs

def save_model_snapshot(model, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(out_dir))
    tokenizer.save_pretrained(str(out_dir))


## 7. Validation reward before RL

In [10]:
@torch.no_grad()
def evaluate_reward(model, rows, max_samples=None, desc='eval_reward'):
    model.eval()
    if max_samples is not None:
        rows = rows[:max_samples]

    preds, rewards, details = [], [], []
    for row in tqdm(rows, desc=desc):
        batch = tokenizer(
            [row['source']],
            max_length=CONFIG['MAX_INPUT_LEN'],
            truncation=True,
            padding=True,
            return_tensors='pt',
        )
        ids = generate_for_rl(model, batch, do_sample=False, num_beams=CONFIG['NUM_BEAMS_EVAL'])
        pred = tokenizer.decode(ids[0], skip_special_tokens=True)
        r, d = reward_one(row['source'], row['summary'], pred)
        preds.append(pred)
        rewards.append(r)
        details.append(d)

    df = pd.DataFrame(details)
    result = {
        'reward': float(np.mean(rewards)),
        'rougeL': float(df['rougeL'].mean()),
        'number_recall': float(df['number_recall'].mean()),
        'number_hallucination': float(df['number_hallucination'].mean()),
        'length_compliance': float(df['length_compliance'].mean()),
        'length_ratio_mean': float(df['length_ratio'].mean()),
    }
    model.train()
    return result, preds

val_rows = [to_pair(r) for r in val_raw]
base_reward_metrics, _ = evaluate_reward(policy, val_rows, max_samples=CONFIG['VAL_REWARD_SAMPLES'], desc='base val reward')
base_reward_metrics


base val reward:   0%|          | 0/32 [00:00<?, ?it/s]

{'reward': 0.6522019451982267,
 'rougeL': 0.7977584656872755,
 'number_recall': 0.850707408910534,
 'number_hallucination': 0.07900042665667666,
 'length_compliance': 0.625,
 'length_ratio_mean': 0.2269491006451107}

In [11]:
# 1. Tải lại Policy Model từ Checkpoint Epoch 1
EPOCH_1_PATH = RL_CHECKPOINT_DIR / "epoch_1"
if not EPOCH_1_PATH.exists():
    raise FileNotFoundError("Không tìm thấy checkpoint Epoch 1 tại Drive. Vui lòng kiểm tra lại đường dẫn.")

print(f"Đang tải lại mô hình từ: {EPOCH_1_PATH}")
policy = AutoModelForSeq2SeqLM.from_pretrained(str(EPOCH_1_PATH)).to(DEVICE)
policy.gradient_checkpointing_enable()
policy.config.use_cache = False

# 2. Khôi phục Reference Model (vẫn lấy từ SFT ban đầu)
ref_model = AutoModelForSeq2SeqLM.from_pretrained(str(SFT_MODEL_DIR)).to(DEVICE)
ref_model.eval()
ref_model.requires_grad_(False)

# 3. Khôi phục lịch sử huấn luyện (nếu có)
history_file = RL_RESULTS_DIR / 'rl_history.json'
if history_file.exists():
    with open(history_file, 'r', encoding='utf-8') as f:
        history = json.load(f)
    best_val_reward = max([h['val_reward'] for h in history])
    # Khôi phục baseline từ cuối Epoch 1 (dựa trên log của bạn là 0.171)
    running_baseline = history[-1].get('train_reward_mean', 0.171)
    print(f"Đã khôi phục lịch sử. Best Val Reward hiện tại: {best_val_reward:.4f}")
else:
    history = []
    best_val_reward = -1e9
    running_baseline = 0.171 # Giá trị mặc định từ log Epoch 1

Đang tải lại mô hình từ: /content/drive/MyDrive/NLP_Final/Output stage 3 vit5 rl/checkpoints/epoch_1
Đã khôi phục lịch sử. Best Val Reward hiện tại: 0.3060


## 8. RL training loop

In [12]:
optimizer = torch.optim.AdamW(policy.parameters(), lr=CONFIG['LR'])
total_update_steps = math.ceil(len(train_loader) * CONFIG['EPOCHS'] / CONFIG['GRAD_ACCUM'])
warmup_steps = max(1, int(total_update_steps * CONFIG['WARMUP_RATIO']))
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)

# Bỏ qua các bước tối ưu hóa của Epoch 1 trong scheduler
# (Giúp tốc độ học không bị nhảy về mức warmup quá cao)
initial_steps = math.ceil(len(train_loader) / CONFIG['GRAD_ACCUM'])
for _ in range(initial_steps):
    scheduler.step()

global_step = len(train_loader) # Giả định đã xong Epoch 1
update_step = initial_steps

policy.train()
# Chỉ chạy từ Epoch 2 (index 1)
for epoch in range(1, CONFIG['EPOCHS']):
    pbar = tqdm(train_loader, desc=f'RL epoch {epoch+1}/{CONFIG["EPOCHS"]}')
    optimizer.zero_grad(set_to_none=True)

    epoch_rewards = []
    epoch_losses = []

    for step, batch in enumerate(pbar, 1):
        with torch.no_grad():
            gen_ids = generate_for_rl(policy, batch, do_sample=CONFIG['DO_SAMPLE'], num_beams=CONFIG['NUM_BEAMS_TRAIN'])
        preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

        rewards, reward_details = reward_batch(batch['source_text'], batch['gold'], preds)
        rewards_t = torch.tensor(rewards, dtype=torch.float32, device=DEVICE)

        batch_reward_mean = float(rewards_t.mean().item())
        if running_baseline is None:
            running_baseline = batch_reward_mean
        else:
            running_baseline = CONFIG['BASELINE_MOMENTUM'] * running_baseline + (1 - CONFIG['BASELINE_MOMENTUM']) * batch_reward_mean

        advantages = rewards_t - running_baseline
        if advantages.numel() > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        policy_logp = sequence_logprobs(policy, batch, gen_ids)
        with torch.no_grad():
            ref_logp = sequence_logprobs(ref_model, batch, gen_ids)

        approx_kl = (policy_logp - ref_logp).mean()
        pg_loss = -(advantages.detach() * policy_logp).mean()
        loss = pg_loss + CONFIG['KL_COEF'] * approx_kl
        (loss / CONFIG['GRAD_ACCUM']).backward()

        epoch_rewards.extend(rewards)
        epoch_losses.append(float(loss.detach().cpu().item()))
        global_step += 1

        if step % CONFIG['GRAD_ACCUM'] == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(policy.parameters(), CONFIG['MAX_GRAD_NORM'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            update_step += 1

        pbar.set_postfix({
            'reward': f'{np.mean(epoch_rewards[-20:]):.3f}',
            'loss': f'{np.mean(epoch_losses[-20:]):.3f}',
            'kl': f'{float(approx_kl.detach().cpu().item()):.4f}',
            'baseline': f'{running_baseline:.3f}',
        })

    # Validation cuối mỗi Epoch
    val_metrics, _ = evaluate_reward(policy, val_rows, max_samples=CONFIG['VAL_REWARD_SAMPLES'], desc=f'val epoch {epoch+1}')
    row = {
        'epoch': epoch + 1,
        'train_reward_mean': float(np.mean(epoch_rewards)),
        'train_loss_mean': float(np.mean(epoch_losses)),
        **{f'val_{k}': v for k, v in val_metrics.items()},
    }
    history.append(row)
    display(pd.DataFrame(history))

    # Lưu Snapshot
    epoch_dir = RL_CHECKPOINT_DIR / f'epoch_{epoch+1}'
    save_model_snapshot(policy, epoch_dir)

    if val_metrics['reward'] > best_val_reward:
        best_val_reward = val_metrics['reward']
        save_model_snapshot(policy, RL_BEST_DIR)
        save_model_snapshot(policy, RL_MODEL_FOR_NEXT_DIR)
        print(f'New best RL model saved tại Epoch {epoch+1} | val_reward: {best_val_reward:.4f}')

    with open(RL_RESULTS_DIR / 'rl_history.json', 'w', encoding='utf-8') as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

clear_cuda()

/tmp/ipykernel_1748/3649359163.py:10: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


RL epoch 2/2:   0%|          | 0/780 [00:00<?, ?it/s]

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


val epoch 2:   0%|          | 0/32 [00:00<?, ?it/s]

,epoch,train_reward_mean,train_loss_mean,val_reward,val_rougeL,val_number_recall,val_number_hallucination,val_length_compliance,val_length_ratio_mean
0,1,0.166854,-0.044177,0.305958,0.640121,0.797090,0.286678,0.1250,0.337125
1,2,0.327292,-0.245819,0.527938,0.719479,0.799462,0.146554,0.1875,0.234325


New best RL model saved tại Epoch 2 | val_reward: 0.5279


## 9. Final evaluation on test

In [13]:
try:
    del policy
except NameError:
    pass
try:
    del ref_model
except NameError:
    pass
clear_cuda()
print('Cleared policy/reference models before final evaluation.')


Cleared policy/reference models before final evaluation.


In [14]:
@torch.no_grad()
def generate_summaries(model, texts, batch_size=1, desc='generate'):
    model.eval()
    outs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = tokenizer(
            texts[i:i+batch_size],
            max_length=CONFIG['MAX_INPUT_LEN'],
            truncation=True,
            padding=True,
            return_tensors='pt',
        ).to(DEVICE)
        ids = model.generate(
            **batch,
            max_length=CONFIG['MAX_TARGET_LEN'],
            num_beams=CONFIG['NUM_BEAMS_EVAL'],
            no_repeat_ngram_size=CONFIG['NO_REPEAT_NGRAM'],
            length_penalty=CONFIG['LENGTH_PENALTY'],
            early_stopping=True,
        )
        outs.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return outs

test_src = ds['test']['source']
test_gold = ds['test']['summary']

sft_model = AutoModelForSeq2SeqLM.from_pretrained(str(SFT_MODEL_DIR)).to(DEVICE)
sft_model.eval()
preds_sft = generate_summaries(sft_model, test_src, batch_size=CONFIG['EVAL_BATCH'], desc='SFT test')
del sft_model
clear_cuda()

rl_model = AutoModelForSeq2SeqLM.from_pretrained(str(RL_BEST_DIR)).to(DEVICE)
rl_model.eval()
preds_rl = generate_summaries(rl_model, test_src, batch_size=CONFIG['EVAL_BATCH'], desc='RL test')
del rl_model
clear_cuda()


SFT test:   0%|          | 0/105 [00:00<?, ?it/s]

RL test:   0%|          | 0/105 [00:00<?, ?it/s]

In [15]:
from bert_score import score as bertscore
import evaluate
rouge_eval = evaluate.load('rouge')

def evaluate_predictions(srcs, golds, preds, tag):
    r = rouge_eval.compute(predictions=preds, references=golds, use_stemmer=False)
    P, R, F1 = bertscore(preds, golds, lang='vi', model_type='xlm-roberta-base', verbose=False)
    nr = [number_recall(s, p) for s, p in zip(srcs, preds)]
    nh = [number_hallucination(s, p) for s, p in zip(srcs, preds)]
    lc = [length_compliance(s, p) for s, p in zip(srcs, preds)]
    lr = [length_ratio(s, p) for s, p in zip(srcs, preds)]
    rewards = [reward_one(s, g, p)[0] for s, g, p in zip(srcs, golds, preds)]
    return {
        'tag': tag,
        'rouge1': round(r['rouge1'] * 100, 2),
        'rouge2': round(r['rouge2'] * 100, 2),
        'rougeL': round(r['rougeL'] * 100, 2),
        'bertscore_f1': round(F1.mean().item() * 100, 2),
        'number_recall': round(np.mean(nr) * 100, 2),
        'number_hallucination': round(np.mean(nh) * 100, 2),
        'length_compliance': round(np.mean(lc) * 100, 2),
        'length_ratio_mean': round(np.mean(lr), 4),
        'reward': round(np.mean(rewards), 4),
    }

metrics_sft = evaluate_predictions(test_src, test_gold, preds_sft, 'SFT')
metrics_rl = evaluate_predictions(test_src, test_gold, preds_rl, 'RL')

df = pd.DataFrame([metrics_sft, metrics_rl]).set_index('tag')
df


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

,rouge1,rouge2,rougeL,bertscore_f1,number_recall,number_hallucination,length_compliance,length_ratio_mean,reward
tag,,,,,,,,,
SFT,84.58,77.38,76.28,94.57,82.06,7.48,26.67,0.2165,0.6073
RL,80.09,71.05,71.25,93.16,78.40,9.61,17.14,0.2120,0.5610


## 10. Export RL results

In [16]:
records = [
    {
        'id': ds['test']['id'][i],
        'source': test_src[i],
        'gold': test_gold[i],
        'pred_sft': preds_sft[i],
        'pred_rl': preds_rl[i],
    }
    for i in range(len(test_src))
]

final = {
    'stage2_model_dir': str(SFT_MODEL_DIR),
    'rl_output_root': str(RL_DIR),
    'rl_best_model_dir': str(RL_BEST_DIR),
    'rl_model_for_next_stage_dir': str(RL_MODEL_FOR_NEXT_DIR),
    'config': CONFIG,
    'history': history,
    'metrics': {'sft': metrics_sft, 'rl': metrics_rl},
    'records': records,
}
with open(RL_RESULTS_DIR / 'final_results_stage3_rl.json', 'w', encoding='utf-8') as f:
    json.dump(final, f, ensure_ascii=False, indent=2)

df.to_csv(RL_RESULTS_DIR / 'metrics_stage3_rl.csv', encoding='utf-8-sig')
print('Saved:', RL_RESULTS_DIR / 'final_results_stage3_rl.json')
print('Saved:', RL_RESULTS_DIR / 'metrics_stage3_rl.csv')


Saved: /content/drive/MyDrive/NLP_Final/Output stage 3 vit5 rl/results/final_results_stage3_rl.json
Saved: /content/drive/MyDrive/NLP_Final/Output stage 3 vit5 rl/results/metrics_stage3_rl.csv
